# Online Retail II — Loading + Feature Engineering Strategy

Goal: clean data load (both Excel sheets), motivated drop rules, feature engineering compatible with the target models (**SARIMAX, Prophet, DeepAR, iTransformer/LSTM**), description embeddings via **Gemini Embedding 2**, unsupervised clustering, and WMAPE strategy.

The dataset is a **UK-based B2B/B2C online retailer** (corporate gift-ware, both luxury and non-luxury), with two Excel sheets: `Year 2009-2010` and `Year 2010-2011`.

## 1. Loading — read BOTH sheets

`pd.read_excel(path)` only reads the first sheet. To read everything and concatenate:

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("Datasets/online_retail_II.xlsx")

# sheet_name=None -> dict {sheet_name: df}; concat preserves origin via SourceSheet
sheets = pd.read_excel(DATA_PATH, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
df = pd.concat(
    [s.assign(SourceSheet=name) for name, s in sheets.items()],
    ignore_index=True,
)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print(df.shape, sorted(sheets.keys()))
df.head(3)

In [ ]:
# Minimal inspection to justify the drop rules
summary = pd.DataFrame({
    "nulls": df.isna().sum(),
    "dtype": df.dtypes.astype(str),
})
print(summary)
print("\nRows per sheet:\n", df["SourceSheet"].value_counts())
print("\nNegative Quantity:", (df["Quantity"] < 0).sum(),
      " | Negative Price:", (df["Price"] < 0).sum(),
      " | Zero Price:",     (df["Price"] == 0).sum())
print("\nCancellation invoices (start with C):", df["Invoice"].str.startswith("C").sum())

## 2. Drop rules — what to remove and why

Goal: a **net weekly demand series per SKU**. Rows to remove fall into three families.

### 2a. Non-products (admin / fees)
`StockCode` isn't always a product. Strip admin codes that pollute the series:

| StockCode | Meaning |
|---|---|
| `POST` | Postage |
| `DOT` | Dotcom postage |
| `M` / `m` | Manual adjustments |
| `D` | Discount |
| `C2` | Carriage |
| `BANK CHARGES` | Bank fees |
| `AMAZONFEE` | Amazon fee |
| `CRUK` | Charity donation |
| `PADS` | Packaging padding |
| `B` | Bad debt adjustment |
| `TEST` / `gift_*` | Test / gift card |

Practical rule: **drop if `StockCode` does not match `^\d{5}[A-Z]?$`** (five digits, optional letter).

### 2b. Cancellations and returns — the important call

Numbers on the dataset: `Negative Quantity = 22 950`, `Invoice C* = 19 494`, `Customer ID null = 243 007 (≈22.8%)`. After dropping C-invoices, **≈3 500 negative rows remain** (~0.33% of the dataset).

**Adopted strategy**: drop `Quantity < 0` from the target, but **first** use those rows to build the `return_rate_Nw` feature (see §3.4-bis). We model *gross demand* in the target, and feed the return signal to the model as a covariate.

**Why not "net via weekly sum"** (looks elegant):
- A purchase in week N + a return in week N+3 ⇒ week N+3 has `y < 0`.
- DeepAR (Negative-Binomial), LightGBM (Tweedie), `log1p` for LSTM/iTransformer **require `y ≥ 0`**. Clipping to 0 wastes the netting *and* introduces bias.
- Only works if purchase and return fall in the same week — a case where dropping or summing is equivalent.

**Why not FIFO matching `(CustomerID, StockCode)`** (looks rigorous):
- 243 007 rows (≈22.8%) have null `Customer ID` ⇒ matching fails on a quarter of the data.
- O(n²) per group on 1M+ rows to recover bias on <0.4% of the dataset ⇒ poor cost/benefit.
- Returns may refer to purchases prior to the dataset start, unmatchable by construction.

**Other prices**:
- `Price < 0` (5 rows): accounting adjustments → drop.
- `Price == 0` (6 202 rows): samples / damages / gifts → drop.

### 2c. Unusable rows
- `InvoiceDate` NaT → drop.
- `StockCode` or `Description` null → drop.
- Sentinel `Description` (`?`, `damaged`, `wrongly coded`, `lost`, `check`) → drop.

### 2d. Outliers — do NOT drop blindly
Huge quantities (e.g. 80 995 units) are **real wholesale orders**, not errors. Don't drop on quantiles — at most apply `log1p` for models that suffer long tails (LSTM/Prophet) and keep them raw for SARIMAX.

In [ ]:
# Reuse the production helper instead of duplicating logic in the notebook.
import sys
sys.path.insert(0, ".")
from src.tools import split_sales_returns, return_rate_features, aggregate_weekly_sku

sales, returns = split_sales_returns(df)
print("raw:", df.shape)
print("sales (target):", sales.shape, " SKUs:", sales["StockCode"].nunique())
print("returns (feature only):", returns.shape)

In [ ]:
# return_rate_4w preview — built from `returns`, no Customer ID needed.
weekly_sku = aggregate_weekly_sku(sales)
rr = return_rate_features(weekly_sku, returns, windows=(4, 13))
rr.head()

## 3. Feature engineering — designed for the 4 model families

The four models want different inputs, but you can build **a single feature store**:

### 3.1 Target
- `y_qty_w` = sum of `Quantity` (sales only, returns excluded) per `(StockCode, Week)`, Monday-aligned.
- `y_rev_w` = sum of `Revenue` (price × qty of sales). Train on `y_qty_w`, evaluate WMAPE on `y_rev_w` (see §6).
- **Reindexing**: per SKU, build the full weekly axis `min(Week)..max(Week)` and `fillna(0)`.

### 3.2 Calendar / time
For **all** models:
- `week_of_year`, `month`, `quarter`, `year`, `is_month_start/end`, `iso_week`.
- UK + retail holidays (Black Friday, Christmas Eve, Boxing Day, Bank Holidays — `holidays` package, country=`GB`).
- Cyclic encoding (`sin/cos` of week_of_year, month) for LSTM/Transformer; for Prophet just pass `country_holidays="GB"`.

### 3.3 Lag / rolling (LightGBM, DeepAR, iTransformer)
- Lags: `[1, 2, 4, 8, 13, 26, 52]` weeks.
- Rolling mean/std/median window `[4, 13, 26]`.
- **ADI** (Average Demand Interval) and **CV²** to classify the series (Syntetos-Boylan: smooth/erratic/intermittent/lumpy).
- `weeks_since_last_sale`, `share_zero_weeks_last_26`.

### 3.4 Price features
- `price_median_sku`, `price_last_w`, `price_pct_change_w`, `price_relative_to_cluster`.
- Promo proxy: `price_w / price_median_52w < 0.85`.

### 3.4-bis Return features (NEW — derived from negative rows dropped from the target)
- `return_rate_4w`, `return_rate_13w`: rolling sum of returns / rolling sum of sales per SKU.
- `is_high_return_sku`: static flag (lifetime return rate > p90).
- Enter as `dynamic_real` in DeepAR and as features in LightGBM/iTransformer. **No Customer ID needed** — works on 100% of the data.

### 3.5 Static categorical (DeepAR / iTransformer `static_cat`)
- `cluster_id` (see §5).
- `country_top1_share` per SKU.
- `mean_basket_size` (proxy for wholesale vs retail).
- `price_tier` (quantiles of `price_median_sku`).

### 3.6 Description embedding (Gemini Embedding 2)
One embedding per `StockCode`, computed on the canonical description. Implementation lives in `src/tools/embeddings.py` (`embed_sku_descriptions`):
1. Per SKU, most-frequent description (mode); fall back to the longest unique value.
2. Pre-clean: uppercase, collapse whitespace.
3. Batch call to `gemini-embedding-2-preview`, `task_type="CLUSTERING"` for clustering use, `"RETRIEVAL_DOCUMENT"` if used as model features.
4. `output_dimensionality=768` (MRL sweet spot; 3072/1536 also recommended by Google).
5. Cached to parquet (`embeddings/sku_desc_emb.parquet`) — never recompute.

In [ ]:
# Description embedding (requires GEMINI_API_KEY in env). Cached on disk after the first run.
# from src.tools import embed_sku_descriptions
# emb = embed_sku_descriptions(
#     sales,
#     cache_path="embeddings/sku_desc_emb.parquet",
#     dim=768,
#     task_type="CLUSTERING",
# )
# emb.head()

## 4. What each model needs

| Model | Granularity | Key inputs | Notes |
|---|---|---|---|
| **SARIMAX** | per-SKU | `y_qty_w` + exog (`price_w`, holiday, `return_rate_4w`) | Top-N SKUs only by volume; otherwise costly |
| **Prophet** | per-SKU | `ds`, `y`, `country_holidays="GB"`, regressor `return_rate_4w` | Robust on seasonality; weak on intermittents |
| **DeepAR** | global | `target` + `time_features` + `static_cat` (cluster, price_tier) + `dynamic_real` (price, holiday, promo, return_rate_4w) + embeddings as `static_real` | Probabilistic; great on intermittents thanks to NegBin |
| **iTransformer / LSTM** | global multivariate | matrix `[N_skus, T, F]` with lag/calendar/price/return_rate + cluster id | Channel-independent (iTransformer) handles large N well |

## 5. Unsupervised clustering — recommended strategy

**Dual goal**: (a) get a `cluster_id` to use as a categorical feature in global models; (b) enable per-cluster models on intermittent SKUs where a global model isn't enough.

### Per-SKU feature vector
Concatenate three families, **each scaled independently** before concat:
1. **Semantic embedding** (Gemini, dim 768 → reduced to 32-64 via UMAP).
2. **Demand profile**: `mean_qty_w`, `std_qty_w`, `ADI`, `CV²`, `share_zero_weeks`, `seasonality_strength` (STL), `trend_strength`, `return_rate_lifetime`.
3. **Commercial profile**: `price_median`, `price_tier`, `mean_basket_size`, `country_uk_share`, `n_unique_customers`.

### Pipeline
```
embeddings (768) ──UMAP(32)──┐
demand profile  ──StandardScaler──┼──► concat ──► HDBSCAN(min_cluster_size=20)
commercial      ──StandardScaler──┘
```

### Why HDBSCAN over KMeans
- KMeans assumes spherical clusters of similar variance: false here (long tail on demand profile).
- HDBSCAN finds variable-density clusters **and** isolates outliers (label `-1`) — the long tail of rare SKUs goes into a "noise bucket" without polluting the others.
- Validation: silhouette on sub-features, **but most importantly** evaluate clusters by whether they improve downstream WMAPE — that's the metric that matters.

### Alternatives
- **GMM** with BIC for K: useful if you want soft assignments as a continuous feature.
- **Hierarchical (Ward)** + cut at K=8–15: ideal if you want *interpretable* clusters (luxury, kitchen, party, vintage…).

### What to do with the clusters
- Pass as `static_cat` to DeepAR (category embedding).
- iTransformer/LSTM: one-hot or a learnable 16-dim embedding.
- SARIMAX/Prophet: do NOT use cluster as a feature; instead **fit one model per cluster** on SKUs with < 26 active weeks of history (cold-start).

## 6. Strategy to minimize WMAPE

$$\text{WMAPE} = \frac{\sum_i |y_i - \hat{y}_i|}{\sum_i |y_i|}$$

Not a plain MAPE average: **high-volume SKUs dominate the metric**.

### 6.1 Train with the right loss
- WMAPE is **scale-free per-series but volume-weighted globally**. Best approximated by **Tweedie** (sparse non-negative) or **MAE / Quantile q=0.5**, **not MSE**.
- DeepAR: `negative-binomial` (default for counts). LightGBM: `objective="tweedie"`, `tweedie_variance_power≈1.2`.
- iTransformer/LSTM: **Huber** or **MAE** on `log1p(qty)`, not MSE.

### 6.2 Problem segmentation
A single global model rarely wins everywhere:
- **Top 20% SKUs by volume** (≈80% of WMAPE by construction): global **DeepAR + iTransformer** ensemble.
- **Mid 60%**: same global model.
- **Tail 20% intermittent**: **Croston-SBA** or **ZIP** per cluster — a Transformer overfits and predicts always 0.

### 6.3 Validation
- **Rolling-origin** (expanding window) with at least 3 folds of 12-week test.
- WMAPE per cluster and per volume tier on top of global WMAPE: aggregate gains can hide regressions on the head.

### 6.4 Ensemble
Errors from SARIMAX/Prophet (local) and DeepAR/iTransformer (global) are **decorrelated**. Weighted average `weights ~ 1/MAE on validation per SKU`: −1 to −3 WMAPE points vs. the best single model.

### 6.5 Final bias correction
Per cluster: $\hat{y}_{final} = a_{c} \cdot \hat{y}_{model} + b_{c}$ with `a, b` fit on validation. Free 0.5–1.5 points.

### 6.6 Sanity baselines
First of all: **seasonal-naive** (`y_t = y_{t-52}`) and **rolling mean 4w**. If a complex model can't beat them on the same volume tier, the bug is in preprocessing.

## 7. Suggested workflow

1. Cleaning + sales/returns split (§2) → `data/processed/sales.parquet`, `data/processed/returns.parquet`.
2. Weekly aggregation per SKU + reindex + `return_rate_Nw` → `data/processed/weekly_sku.parquet`.
3. Gemini description embeddings → `data/processed/sku_desc_emb.parquet` (one-time).
4. Demand + commercial profile → `data/processed/sku_static.parquet`.
5. Clustering (§5) → adds `cluster_id` to `sku_static`.
6. Baselines (seasonal-naive, mean) for reference WMAPE.
7. SARIMAX top-N + Prophet → statistical benchmarks.
8. DeepAR global → first "serious" model.
9. iTransformer/LSTM → second global model for the ensemble.
10. Per-cluster bias correction + ensemble.